# conv_system on the PYNQ-Z2

Streaming N×N convolution accelerator (IEEE SSCS Egypt 2026). The Zynq processing system loads a kernel and streams a frame through the AXI4-Lite wrapper `conv_axi_lite`; every output feature is read back and compared with the bit-accurate golden model.

The driver resets the accelerator once when the overlay is opened; every frame below streams after the previous one **without a reset**. `conv_system` marks the last feature of each frame with `o_frame_done`, which sets the wrapper's frame-done flag and `FRAME_COUNT`; a kernel is rewritten only when it changes.

Files next to this notebook: `conv_demo.bit`, `conv_demo.hwh` (from `fpga/bitstream/`) and `conv_demo.py` (driver + golden model).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from conv_demo import ConvAccelerator, DEMO_KERNELS, pattern, load_image, reference

accel = ConvAccelerator()          # downloads conv_demo.bit
print('image %dx%d, kernel %dx%d, %d lane(s), ReLU %s' % (accel.input_size, accel.input_size,
      accel.window, accel.window, accel.num_kernels, accel.relu))

## One frame: checkerboard through Sobel-X

In [ ]:
image = pattern('checker', accel.input_size)
kernel = DEMO_KERNELS['sobel_x']
hw, stats = accel.run(image, [kernel])
ref = reference(image, kernel, accel.relu)
print('mismatching features:', np.count_nonzero(hw[0] != ref))
print(stats)

fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
for a, img, title in zip(ax, (image, hw[0], hw[0] - ref), ('input', 'FPGA output', 'FPGA - model')):
    a.imshow(img, cmap='gray'); a.set_title(title); a.axis('off')
plt.show()

## Every demo kernel on every pattern

In [ ]:
patterns = ['bars', 'checker', 'circle', 'ramp', 'noise']
names = ['identity', 'sobel_x', 'sobel_y', 'laplacian', 'gaussian', 'sharpen', 'emboss']
fig, ax = plt.subplots(len(patterns), len(names) + 1, figsize=(2 * (len(names) + 1), 2.1 * len(patterns)))
total_bad = 0
for r, p in enumerate(patterns):
    image = pattern(p, accel.input_size)
    ax[r][0].imshow(image, cmap='gray', vmin=0, vmax=255); ax[r][0].set_title(p)
    for c, name in enumerate(names, start=1):
        hw, stats = accel.run(image, [DEMO_KERNELS[name]])
        bad = np.count_nonzero(hw[0] != reference(image, DEMO_KERNELS[name], accel.relu))
        total_bad += bad
        ax[r][c].imshow(hw[0], cmap='gray')
        ax[r][c].set_title('%s %s' % (name, 'ok' if bad == 0 else '%d bad' % bad), fontsize=9)
for a in ax.flat:
    a.axis('off')
plt.tight_layout(); plt.show()
print('total mismatching features:', total_bad)

## Edge detection on a picture

Upload any image next to the notebook and set `path`. The picture is converted to grey and resized to the overlay's input size; the edge magnitude combines the Sobel-X and Sobel-Y frames computed on the FPGA.

In [ ]:
path = 'photo.jpg'
try:
    image = load_image(path, accel.input_size)
except (IOError, OSError):
    print('%s not found, using the circle pattern' % path)
    image = pattern('circle', accel.input_size)

gx, _ = accel.run(image, [DEMO_KERNELS['sobel_x']])
gy, _ = accel.run(image, [DEMO_KERNELS['sobel_y']])
magnitude = np.hypot(gx[0].astype(float), gy[0].astype(float))

fig, ax = plt.subplots(1, 4, figsize=(13, 3.5))
for a, img, title in zip(ax, (image, gx[0], gy[0], magnitude), ('input', 'Sobel-X (FPGA)', 'Sobel-Y (FPGA)', 'edge magnitude')):
    a.imshow(img, cmap='gray'); a.set_title(title); a.axis('off')
plt.show()

## Timing

`latency_cycles` counts clock cycles from the first pixel to the first feature inside the fabric. The wall-clock time is dominated by the AXI4-Lite register writes from Python, not by the accelerator, which produces one feature per clock cycle once the output FIFO is primed.

The ten frames run back to back on the same kernel with no reset and no kernel write in between; `frames_since_reset` counts every frame completed since the overlay was opened.

In [ ]:
import time
image = pattern('noise', accel.input_size)
runs = 10
t0 = time.time()
for _ in range(runs):
    hw, stats = accel.run(image, [DEMO_KERNELS['laplacian']])
elapsed = (time.time() - t0) / runs
print('frame incl. AXI transfers: %.1f ms (%.0f frames/s)' % (1000 * elapsed, 1 / elapsed))
print('in-fabric latency: %d cycles, first pixel to last feature: %d cycles' % (stats['latency_cycles'], stats['frame_cycles']))
print('frames completed since reset:', stats['frames_since_reset'])